## Data descriptions

---
In this competition you are predicting the probability that an online transaction is fraudulent, as denoted by the binary target isFraud.

The data is broken into two files identity and transaction, which are joined by TransactionID. Not all transactions have corresponding identity information.

Categorical Features - Transaction
ProductCD
card1 - card6
addr1, addr2
P_emaildomain
R_emaildomain
M1 - M9
Categorical Features - Identity
DeviceType
DeviceInfo
id_12 - id_38
The TransactionDT feature is a timedelta from a given reference datetime (not an actual timestamp).

You can read more about the data from this post by the competition host.

Files
train_{transaction, identity}.csv - the training set
test_{transaction, identity}.csv - the test set (you must predict the isFraud value for these observations)
sample_submission.csv - a sample submission file in the correct format

---

I see many questions regarding data description, so it maybe a better idea to open a thread for discussion. The following is a bit more details about it:

Transaction Table *

TransactionDT: timedelta from a given reference datetime (not an actual timestamp)

TransactionAMT: transaction payment amount in USD

ProductCD: product code, the product for each transaction

card1 - card6: payment card information, such as card type, card category, issue bank, country, etc.

addr: address

dist: distance

P_ and (R__) emaildomain: purchaser and recipient email domain

C1-C14: counting, such as how many addresses are found to be associated with the payment card, etc. The actual meaning is masked.

D1-D15: timedelta, such as days between previous transaction, etc.

M1-M9: match, such as names on card and address, etc.

Vxxx: Vesta engineered rich features, including ranking, counting, and other entity relations.

Categorical Features: ProductCD card1 - card6 addr1, addr2 P_emaildomain R_emaildomain M1 - M9

Identity Table *

Variables in this table are identity information – network connection information (IP, ISP, Proxy, etc) and digital signature (UA/browser/os/version, etc) associated with transactions. They're collected by Vesta’s fraud protection system and digital security partners. (The field names are masked and pairwise dictionary will not be provided for privacy protection and contract agreement)

Categorical Features: DeviceType DeviceInfo id_12 - id_38

---

In [175]:
import pandas as pd
import networkx as nx

In [176]:
train_identity = pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction = pd.read_csv('ieee-fraud-detection/train_transaction.csv')


In [177]:
train_identity.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [178]:
train_transaction.head().iloc[:, :12]


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0


# Plan for making a graph
- Many nodes: eamils, transaction id, addresses, devices, IP information (identify all of these in the two data tables and work on relating the in networkx)
- Make all edges connections from transaction to other node types

``` 
Basic example to build from
import networkx as nx

G = nx.Graph()

for _, row in df.iterrows():
    t = f"T_{row['TransactionID']}"
    card = f"C_{row['card1']}"
    
    G.add_edge(t, card)
```

In [179]:
train_identity[['id_01', 'id_02']]

,id_01,id_02
0,0.0,70787.0
1,-5.0,98945.0
2,-5.0,191631.0
3,-5.0,221832.0
4,0.0,7460.0
...,...,...
144228,-15.0,145955.0
144229,-5.0,172059.0
144230,-20.0,632381.0
144231,-5.0,55528.0


In [180]:
def create_graph_edges_for_similar_cols(
        target_graph: nx.Graph,
        target_df: pd.DataFrame, 
        primary_node_col: str,
        primary_node_prefix: str,
        target_col_names: list[str], 
        ref_node_prefix: str, 
        ref_node_suffix: list[str] = None
    ) -> int:
    """
        This is a utility function for iterating similar columns that might need to be added to a graph in bulk. It will edit the graph passed in the argument directly.
        Retruns the size of the updated graph.

        graph: An instance of a networkx graph that needs to have connections added to it
        target_df: The data frame containing all reference columns for the mappings
        primary_node_col: The name of the column that will server as the primary node that should as the basis for many connections (ex. Transactions in a case where you want to connect many other attributes about that transaction to it),
        primary_node_prefix: A string that is the prefix for the primary node that will be used on the Graph (ex. if the primary node is Transactions you may want to use something like 'T' or 'Tr')
        target_col_names: a list of strings containing the names of columns in the target DataFrame that need to have relationships created for
        graph_label_prefix: the desired prefix for the target column to be associated with in the graph (ex if the columns are all related to credit card info you may want to use somehting like 'C')
        graph_label_suffix: this is the method for appending unique identifiers to the graph_label_prefix when there are multiple cols (once again for card info you may want something like C1, C2 ...)
                            If nothing is passed to this argument it will fill with 1 - n where n is the len of the  target_col_names. If you want custom functionality pass list of the suffix values in order.
    """
    tdf = target_df.copy() # ensure that the orginal df is not edited by this

    if ref_node_suffix != None:
        assert len(set(ref_node_suffix)) == len(ref_node_suffix), 'Graph label suffix values must be unique.'
    else: 
        ref_node_suffix = list(range(1, len(target_col_names) + 1)) # default functionality for using digits as the suffix

    for col_name, suffix in zip(target_col_names, ref_node_suffix):
        mask = tdf[col_name].notna()
        
        target_data = tdf[mask]

        connection_tuple_mappings = zip(
            f'{primary_node_prefix}_' + target_data[primary_node_col].astype('str'), 
            f'{ref_node_prefix}{suffix}_' + target_data[col_name].astype('str') # puts creates prefix of C1, C2, ... to C6 so each card identifier is unique
        )

        target_graph.add_edges_from(connection_tuple_mappings)
    
    return target_graph.size()


In [181]:
# get all connections created
# zipping the target cols as string and using add_edges_from() is more efficent so use this

# instantiate the graph
transactions_graph = nx.Graph()

# get all relevant fields that need to be added to the graph (all of this centers on transaction IDs and attributes that relate to those)
# all card data (card1 - card6)
card_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6']

# create card connections
create_graph_edges_for_similar_cols(
    target_graph= transactions_graph,
    target_df=train_transaction,
    primary_node_col='TransactionID',
    primary_node_prefix='Tr',
    target_col_names=card_cols,
    ref_node_prefix='C'
)

addr_cols = ['addr1', 'addr2']
# address info (addr1 and addr2 )
create_graph_edges_for_similar_cols(
    target_graph= transactions_graph,
    target_df=train_transaction,
    primary_node_col='TransactionID',
    primary_node_prefix='Tr',
    target_col_names=addr_cols,
    ref_node_prefix='Addr'
)

# identity information is next
# Looks like id_02 is the only thing that can reasonably be used without introducing a bunch of nodes with artificially high degree measures
create_graph_edges_for_similar_cols(
    target_graph= transactions_graph,
    target_df=train_identity,
    primary_node_col='TransactionID',
    primary_node_prefix='Tr',
    target_col_names=['id_02'],
    ref_node_prefix='Id'
)



4715875

In [182]:
# inspecting basic graph attributes
print(f'Number of edges: {transactions_graph.number_of_edges()}')
print(f'Number of ndoes:  {transactions_graph.number_of_nodes()}')

Number of edges: 4715875
Number of ndoes:  720895


In [183]:
from collections import Counter

# looking at the nodes by each column from the original data set
Counter(n.split('_')[0] for n in transactions_graph.nodes)

Counter({'Tr': 590540,
         'Id1': 115655,
         'C1': 13553,
         'C2': 500,
         'Addr1': 332,
         'C5': 119,
         'C3': 114,
         'Addr2': 74,
         'C4': 4,
         'C6': 4})

In [184]:
# analyzing the degree structure of the nodes

In [185]:
nodes_sorted_by_degree = sorted(list(transactions_graph.degree), key=lambda x: x[1], reverse=True)
nodes_sorted_by_degree[:20]

[('C3_150.0', 521287),
 ('Addr2_87.0', 520481),
 ('C6_debit', 439938),
 ('C4_visa', 384767),
 ('C5_226.0', 296546),
 ('C4_mastercard', 189217),
 ('C6_credit', 148986),
 ('C5_224.0', 81513),
 ('C5_166.0', 57140),
 ('C3_185.0', 56346),
 ('C2_321.0', 48935),
 ('Addr1_299.0', 46335),
 ('C2_111.0', 45191),
 ('Addr1_325.0', 42751),
 ('Addr1_204.0', 42020),
 ('C2_555.0', 41995),
 ('Addr1_264.0', 39870),
 ('C2_490.0', 38145),
 ('C5_102.0', 29105),
 ('Addr1_330.0', 26287)]

- It looks like C4 and C6 values may not be very useful here since they are generic categorical classifications that are likely to be shared and connect too many values.
- Will coninue to explore non-transaction nodes for high connectivity

In [186]:
nodes_to_explore = []

for pair in nodes_sorted_by_degree:
    non_interesting_nodes = ['Tr', 'C4', 'C6']
    node = pair[0]

    if node[:2] in non_interesting_nodes: continue

    nodes_to_explore.append(pair)


In [187]:
nodes_to_explore[:20]

[('C3_150.0', 521287),
 ('Addr2_87.0', 520481),
 ('C5_226.0', 296546),
 ('C5_224.0', 81513),
 ('C5_166.0', 57140),
 ('C3_185.0', 56346),
 ('C2_321.0', 48935),
 ('Addr1_299.0', 46335),
 ('C2_111.0', 45191),
 ('Addr1_325.0', 42751),
 ('Addr1_204.0', 42020),
 ('C2_555.0', 41995),
 ('Addr1_264.0', 39870),
 ('C2_490.0', 38145),
 ('C5_102.0', 29105),
 ('Addr1_330.0', 26287),
 ('C5_117.0', 25941),
 ('Addr1_315.0', 23078),
 ('C2_583.0', 21803),
 ('Addr1_441.0', 20827)]

In [188]:
comps = list(nx.connected_components(transactions_graph))
# [len(c) for c in comps]
sorted([len(c) for c in comps], reverse=True)
comps[1:]

[{'C1_13708', 'Id1_334674.0', 'Tr_3301059'},
 {'C1_3823', 'Id1_263916.0', 'Tr_3513119'}]

- Obviously there is some node that is connecting eeryhting leadig to only two components that are not connected to the rest.
- Will investigate the large component further using the highest degree nodes that aren't transactions from nodes_to_explore from earlier.

In [189]:
nodes_to_explore[:20]

[('C3_150.0', 521287),
 ('Addr2_87.0', 520481),
 ('C5_226.0', 296546),
 ('C5_224.0', 81513),
 ('C5_166.0', 57140),
 ('C3_185.0', 56346),
 ('C2_321.0', 48935),
 ('Addr1_299.0', 46335),
 ('C2_111.0', 45191),
 ('Addr1_325.0', 42751),
 ('Addr1_204.0', 42020),
 ('C2_555.0', 41995),
 ('Addr1_264.0', 39870),
 ('C2_490.0', 38145),
 ('C5_102.0', 29105),
 ('Addr1_330.0', 26287),
 ('C5_117.0', 25941),
 ('Addr1_315.0', 23078),
 ('C2_583.0', 21803),
 ('Addr1_441.0', 20827)]

In [190]:
# looking at transactions that are associated with this highly connected address
list(transactions_graph.neighbors('Addr1_299.0'))[:20]

['Tr_2987020',
 'Tr_2987022',
 'Tr_2987039',
 'Tr_2987045',
 'Tr_2987050',
 'Tr_2987058',
 'Tr_2987065',
 'Tr_2987081',
 'Tr_2987089',
 'Tr_2987109',
 'Tr_2987112',
 'Tr_2987116',
 'Tr_2987119',
 'Tr_2987123',
 'Tr_2987124',
 'Tr_2987127',
 'Tr_2987149',
 'Tr_2987155',
 'Tr_2987159',
 'Tr_2987163']

In [191]:
# exploring the neighbors of one transaction
list(transactions_graph.neighbors('Tr_2987020'))

['C1_7875',
 'C2_314.0',
 'C3_150.0',
 'C4_mastercard',
 'C5_224.0',
 'C6_debit',
 'Addr1_299.0',
 'Addr2_87.0']

In [192]:
# getting the 2 neighbor out sub graph for the same transaction
sub_graph = nx.ego_graph(transactions_graph, 'Tr_2987020', radius=2)

In [193]:
# inspecting the size of the sub graph
len(list(sub_graph)) 

575640

- It is remarkable how quickly this ballooned after the addition of one additional neighbor since all nodes associated witht this connection also have a high degree.

### Moving on to getting the transaction to transaction relationships worked out

In [194]:
# trimming the graph down to remove connections that are too deep to make bipartite transformation easier
trimmed_graph = transactions_graph.copy()

nodes_to_trim = [pair for pair in nodes_sorted_by_degree if not pair[0].startswith('Tr')]
for pair in nodes_to_trim:
    if pair[1] > 100:
        try:
            trimmed_graph.remove_node(pair[0])
        except: continue

In [195]:
from networkx.algorithms import bipartite

# fitlering down to just transaction nodes
trimmed_nodes = [node for node in trimmed_graph.nodes if node.startswith('Tr')]


# turn this into a bipartite graph where transaction nodes are all directly connected if they share other connection nodes associated with attributes
# need to do some reduction to the graph before running this, it is taking way too long
trans_to_trans_graph = bipartite.weighted_projected_graph(trimmed_graph, trimmed_nodes)

In [196]:
# looking at the most connected transactions
# weight of the edge is a direct indicaition of shared features here
sorted(
    trans_to_trans_graph.edges(data=True),
    key=lambda x: x[2]['weight'],
    reverse=True
)[:50]

[('Tr_3231907', 'Tr_3231926', {'weight': 5}),
 ('Tr_3231907', 'Tr_3231911', {'weight': 5}),
 ('Tr_3231911', 'Tr_3231926', {'weight': 5}),
 ('Tr_3004561', 'Tr_3012016', {'weight': 4}),
 ('Tr_3004561', 'Tr_3004571', {'weight': 4}),
 ('Tr_3004561', 'Tr_3012035', {'weight': 4}),
 ('Tr_3004561', 'Tr_3004589', {'weight': 4}),
 ('Tr_3004561', 'Tr_3004623', {'weight': 4}),
 ('Tr_3004571', 'Tr_3012016', {'weight': 4}),
 ('Tr_3004571', 'Tr_3012035', {'weight': 4}),
 ('Tr_3004571', 'Tr_3004589', {'weight': 4}),
 ('Tr_3004571', 'Tr_3004623', {'weight': 4}),
 ('Tr_3004589', 'Tr_3012016', {'weight': 4}),
 ('Tr_3004589', 'Tr_3012035', {'weight': 4}),
 ('Tr_3004589', 'Tr_3004623', {'weight': 4}),
 ('Tr_3004623', 'Tr_3012016', {'weight': 4}),
 ('Tr_3004623', 'Tr_3012035', {'weight': 4}),
 ('Tr_3009994', 'Tr_3010111', {'weight': 4}),
 ('Tr_3009994', 'Tr_3010002', {'weight': 4}),
 ('Tr_3010002', 'Tr_3010111', {'weight': 4}),
 ('Tr_3010656', 'Tr_3010706', {'weight': 4}),
 ('Tr_3011390', 'Tr_3011417', {'we

In [197]:
sorted(list(trans_to_trans_graph.degree()), key=lambda x: x[1], reverse=True)

[('Tr_3231911', 231),
 ('Tr_3231907', 230),
 ('Tr_3231926', 230),
 ('Tr_3021585', 186),
 ('Tr_3021643', 186),
 ('Tr_3023706', 186),
 ('Tr_3010766', 183),
 ('Tr_3031080', 182),
 ('Tr_3031088', 182),
 ('Tr_3014200', 178),
 ('Tr_3015164', 178),
 ('Tr_3347905', 178),
 ('Tr_3011956', 176),
 ('Tr_3057921', 176),
 ('Tr_3169705', 176),
 ('Tr_3169722', 176),
 ('Tr_3338596', 176),
 ('Tr_3057918', 175),
 ('Tr_3080639', 175),
 ('Tr_3080641', 175),
 ('Tr_3169711', 175),
 ('Tr_3169713', 175),
 ('Tr_3195485', 175),
 ('Tr_3013075', 174),
 ('Tr_3044348', 174),
 ('Tr_3014165', 173),
 ('Tr_3269745', 173),
 ('Tr_3013939', 172),
 ('Tr_3044349', 172),
 ('Tr_3046642', 172),
 ('Tr_3046652', 172),
 ('Tr_3093775', 172),
 ('Tr_3245944', 172),
 ('Tr_3417345', 172),
 ('Tr_3158125', 170),
 ('Tr_3443816', 170),
 ('Tr_3011566', 168),
 ('Tr_3061564', 168),
 ('Tr_3101695', 168),
 ('Tr_3321316', 168),
 ('Tr_3021885', 163),
 ('Tr_3036732', 163),
 ('Tr_3204864', 163),
 ('Tr_3470692', 163),
 ('Tr_3500867', 163),
 ('Tr_3544

In [198]:
# adding in fraudulent tranasctions so that it is also included in the graph

fraud_mappings = {
    f'Tr_{trans_id}': fraud_status
    for trans_id, fraud_status in zip(
        train_transaction.TransactionID,
        train_transaction.isFraud
    )
}

# setting fraud values on the graph transaction nodes
nx.set_node_attributes(trans_to_trans_graph, fraud_mappings, 'fraud_status')

In [244]:
# will explore the fraud ratios on the top 15 most connected nodes
nodes_of_interest = sorted(list(trans_to_trans_graph.degree()), key=lambda x: x[1], reverse=True)[:15]

In [250]:
# looking at the neighbors of the most connected transaction
# neighbors = trans_to_trans_graph.neighbors('Tr_2987288')
node_of_interest = 'Tr_2987288'

# keep list of tupeles of (node, fraud_count, neighbors, ratio)
fraud_counts = []

# get fraud counts for all nodes of interest
for node, total_connections in nodes_of_interest:
    # exploring the fraud count of the neighbors of this transaction
    fraud_count = sum(
        trans_to_trans_graph.nodes[n].get('fraud_status', 0)
        for n in trans_to_trans_graph.neighbors(node)  
    )
    
    fraud_counts.append((node, fraud_count, total_connections, (round(fraud_count / total_connections, 4))))



In [251]:
fraud_counts


[('Tr_3231911', 3, 231, 0.013),
 ('Tr_3231907', 3, 230, 0.013),
 ('Tr_3231926', 3, 230, 0.013),
 ('Tr_3021585', 6, 186, 0.0323),
 ('Tr_3021643', 6, 186, 0.0323),
 ('Tr_3023706', 6, 186, 0.0323),
 ('Tr_3010766', 6, 183, 0.0328),
 ('Tr_3031080', 0, 182, 0.0),
 ('Tr_3031088', 0, 182, 0.0),
 ('Tr_3014200', 6, 178, 0.0337),
 ('Tr_3015164', 6, 178, 0.0337),
 ('Tr_3347905', 6, 178, 0.0337),
 ('Tr_3011956', 7, 176, 0.0398),
 ('Tr_3057921', 7, 176, 0.0398),
 ('Tr_3169705', 7, 176, 0.0398)]

### Feature creation next
- degree
- weighted_degree
- component_size
- neighbor_fraud_ratio
- average_neighbor_degree
- average_edge_weight
- clustering_coefficient
- pagerank

In [253]:
# degree
t2t_degree_dict = dict(trans_to_trans_graph.degree())

# weighted degrees
t2t_w_degree_dict = dict(trans_to_trans_graph.degree(weight='weight'))

# clustering coef.: looks at how connected interconnected neighbors are to a given node
t2t_clustering_coef_dict = nx.clustering(trans_to_trans_graph, weight='weight')

# page rank: details how influential a node is on it neighborhood
t2t_page_rank_dict = nx.pagerank(trans_to_trans_graph, weight='weight')


In [257]:
nx.to_scipy_sparse_array(trans_to_trans_graph)

<Compressed Sparse Row sparse array of dtype 'int64'
	with 4912860 stored elements and shape (590540, 590540)>

In [ ]:
# manually implemented graph features

# connected component size
comp_sizes = {}
for comp in nx.connected_components(trans_to_trans_graph):
    size = len(comp)
    for node in comp:
        comp_sizes[node] = size

# neighbor fraud ratio
# this needs to be done by turning the edges dict in to a dataframe and then merging the lable info onto it from train_transaction
# # edges = pd.DataFrame(G.edges(), columns=['src', 'dst'])
# this is also undirected so dont forget to add al revers realtions too before calculating ratios
# edges_rev = edges.rename(
#     columns={'src':'dst', 'dst':'src'}
# )

# edges_full = pd.concat([edges, edges_rev])

# average edge weight
# this will be easier to calculate directly on the df since it is just (weighted degree / degree)
